In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score

In [21]:
df = pd.read_csv("Sleep_health_and_lifestyle_dataset.csv", encoding='latin1')
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


In [22]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Person ID                374 non-null    int64  
 1   Gender                   374 non-null    object 
 2   Age                      374 non-null    int64  
 3   Occupation               374 non-null    object 
 4   Sleep Duration           374 non-null    float64
 5   Quality of Sleep         374 non-null    int64  
 6   Physical Activity Level  374 non-null    int64  
 7   Stress Level             374 non-null    int64  
 8   BMI Category             374 non-null    object 
 9   Blood Pressure           374 non-null    object 
 10  Heart Rate               374 non-null    int64  
 11  Daily Steps              374 non-null    int64  
 12  Sleep Disorder           155 non-null    object 
dtypes: float64(1), int64(7), object(5)
memory usage: 38.1+ KB


Person ID                    0
Gender                       0
Age                          0
Occupation                   0
Sleep Duration               0
Quality of Sleep             0
Physical Activity Level      0
Stress Level                 0
BMI Category                 0
Blood Pressure               0
Heart Rate                   0
Daily Steps                  0
Sleep Disorder             219
dtype: int64

In [23]:
#2. Data Preprocessing
#Removing the missing values

In [24]:
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].mean())
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [25]:
df.isnull().sum()

Person ID                  0
Gender                     0
Age                        0
Occupation                 0
Sleep Duration             0
Quality of Sleep           0
Physical Activity Level    0
Stress Level               0
BMI Category               0
Blood Pressure             0
Heart Rate                 0
Daily Steps                0
Sleep Disorder             0
dtype: int64

In [26]:
#Splitting the dataset

In [27]:
X = df.drop(['Person ID', 'Stress Level'], axis = 1)
y = df['Stress Level']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, random_state=42
)
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (243, 11)
Testing set shape: (131, 11)


In [28]:
#Encoding categorical variables

In [29]:
categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(include=[np.number]).columns

categorical_cols, numerical_cols

(Index(['Gender', 'Occupation', 'BMI Category', 'Blood Pressure',
        'Sleep Disorder'],
       dtype='object'),
 Index(['Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level',
        'Heart Rate', 'Daily Steps'],
       dtype='object'))

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ])

In [31]:
#Scaling the numerical features

In [32]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
scaler = StandardScaler(with_mean=False)  # with_mean=False because of sparse matrix
X_train_scaled = scaler.fit_transform(X_train_processed)
X_test_scaled = scaler.transform(X_test_processed)


# Linear Regression

In [33]:
#train the model using train dataset
linreg = LinearRegression() 
linreg.fit(X_train_scaled, y_train) 

#predicted values of the y training data set
y_train_pred = linreg.predict(X_train_scaled) 

#apply yhe model to the test data set
y_test_pred = linreg.predict(X_test_scaled) 

## Goodness of fit of the Linear Regression model

In [34]:
# Check the Goodness of Fit (on Train Data)
print("Goodness of Fit of Linear Regression Model Train Dataset")
print("Explained Variance (R^2):", linreg.score(X_train_scaled, y_train))
print("Mean Squared Error (MSE):", mean_squared_error(y_train, y_train_pred)) 
print()
 
# Check the Goodness of Fit (on Test Data)
print("Goodness of Fit of Linear Regression Model Test Dataset")
print("Explained Variance (R^2):", linreg.score(X_test_scaled, y_test))
print("Mean Squared Error (MSE):", mean_squared_error(y_test, y_test_pred))
print()

Goodness of Fit of Linear Regression Model Train Dataset
Explained Variance (R^2): 0.9889753853206403
Mean Squared Error (MSE): 0.03403704387433045

Goodness of Fit of Linear Regression Model Test Dataset
Explained Variance (R^2): 0.8772760467719177
Mean Squared Error (MSE): 0.3941240318341629



High R^2 on both train (0.99) and test (0.88) showing that model captures strong linear relationships and can explain most of the variation in the data.
MSE is small in both cases shows that predictions are generally close to their actual values.

However, there’s a drop in R² from training (0.99) to test (0.88).
This suggests slight overfitting where the model performs a bit better on known data than unseen data.
The gap is not big, so the model still generalizes well.

# Ridge Regression with different alpha values

In [35]:
# Ridge Regression model 1
for x in [1, 2, 3, 4, 6, 7, 8, 9]:
    ridge_modelx = Ridge(alpha = x)  # alpha is the regularization strength
    ridge_modelx.fit(X_train_scaled, y_train)
    
    # Make predictions on the train set
    y_train_pred = ridge_modelx.predict(X_train_scaled)
    
    # Make predictions on the test set
    y_test_pred = ridge_modelx.predict(X_test_scaled)

    #Check the goodness of fit of Ridge regression models with different alpha values: 
    print('Ridge model' ,x,':', 'alpha =', x)
    # Check the Goodness of Fit (on Train Data)
    print('Train')
    print("Explained Variance (R^2):", ridge_modelx.score(X_train_scaled, y_train))
    print("Mean Squared Error (MSE):", mean_squared_error(y_train, y_train_pred)) 
    print()
     
    # Check the Goodness of Fit (on Test Data)
    print('Test')
    print("Explained Variance (R^2):", ridge_modelx.score(X_test_scaled, y_test))
    print("Mean Squared Error (MSE):", mean_squared_error(y_test, y_test_pred))
    print()

Ridge model 1 : alpha = 1
Train
Explained Variance (R^2): 0.9882051863095219
Mean Squared Error (MSE): 0.03641493174577542

Test
Explained Variance (R^2): 0.9152561065954153
Mean Squared Error (MSE): 0.2721522902694171

Ridge model 2 : alpha = 2
Train
Explained Variance (R^2): 0.9872149113295946
Mean Squared Error (MSE): 0.039472275146859886

Test
Explained Variance (R^2): 0.9292253471459393
Mean Squared Error (MSE): 0.2272905231684046

Ridge model 3 : alpha = 3
Train
Explained Variance (R^2): 0.986312397317629
Mean Squared Error (MSE): 0.04225866813345397

Test
Explained Variance (R^2): 0.9373763034802652
Mean Squared Error (MSE): 0.20111398884654877

Ridge model 4 : alpha = 4
Train
Explained Variance (R^2): 0.98548943163149
Mean Squared Error (MSE): 0.04479946615505055

Test
Explained Variance (R^2): 0.9423978698188105
Mean Squared Error (MSE): 0.18498738992749353

Ridge model 6 : alpha = 6
Train
Explained Variance (R^2): 0.9840248675517138
Mean Squared Error (MSE): 0.049321114601725

As the alpha value increases,
Training R^2 slightly decreases as regularisation limits model flexibility.
Test R^2 increases and difference of R^2 values between train and test set as overfitting reduces.
MSE also decreases on the test set showing a better generalisation.

Since alpha = 9 gives us the closest values of R^2 for the traing and testing dataset out of all the Ridge Models, we will use ridge model 9 for cross validation.

# k-fold Cross Validation

In [36]:
#Recombine X and y datasets
from scipy.sparse import vstack
X_full = vstack([X_train_scaled, X_test_scaled])
y_full = pd.concat([y_train, y_test], axis=0)

In [37]:
#k-fold 
kf = KFold(n_splits = 5, shuffle = True, random_state = 42) #5-fold validation
linear_scores = cross_val_score(linreg, X_full, y_full, cv = kf, scoring = 'r2')
ridge_scores  = cross_val_score(ridge_modelx, X_full, y_full , cv = kf, scoring = 'r2')

# Calculate the mean and standard deviation of R² values for each model
print("Linear Regression: Mean R² =", np.mean(linear_scores), " | Standard deviation =", np.std(linear_scores))
print("Ridge Regression : Mean R² =", np.mean(ridge_scores),  " | Standard deviation =", np.std(ridge_scores))

#If ridge regression has larger average R2 and smaller standard deviation, it
#means Ridge gives slightly better and more stable performance — the regularization helps reduce overfitting and variance.

Linear Regression: Mean R² = 0.8784582270486997  | Standard deviation = 0.15382271943989376
Ridge Regression : Mean R² = 0.9605003112942745  | Standard deviation = 0.013819701586639192


Therefore, we can see that the Ridge regression has better and more stable performance because it has higher values and lower standard deviations for its R² score

Linear Regression is more suitable when simplicity and interpretability are priorities and datasets are small with clear linear relationships. -> Predicting material stress/strain under simple conditions

Ridge Regression is more suitable when predictions stability and generalisation are more important than exact interpretability, especially with datasets that are high-dimensional and multicollinearity exists or noisy data.
-> Predicting stock returns using correlated economic indicators.